In [1]:
import tensorflow as tf 
from tensorflow.keras.models import load_model
import pickle 
import pandas as pd
import numpy as np 

In [2]:
model = load_model('model.h5')
with open('onehot_encoder_geo.pkl','rb') as file:
    onehot_encoder_geo = pickle.load(file)
with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)
with open('scaler.pkl','rb') as file:
    scalar = pickle.load(file)

In [5]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 600000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [7]:
#onehotencoded geo 
geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded,columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

c:\Users\bhava\annclassification\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [12]:
#labelencoded gender
gender_encoded = label_encoder_gender.transform([input_data['Gender']])
gender_encoded_df = pd.DataFrame(gender_encoded,columns=['Gender'])

In [13]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [14]:
gender_encoded_df

,Gender
0,1


In [16]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,600000,2,1,1,50000


In [18]:
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df

ValueError: y contains previously unseen labels: 1

In [19]:
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,600000,2,1,1,50000


In [20]:
input_df = pd.concat([input_df.drop("Geography",axis=1), geo_encoded_df], axis=1)

In [21]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,600000,2,1,1,50000,1.0,0.0,0.0


In [22]:
input_scaled = scalar.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349,  8.38812313,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [23]:
prediction = model.predict(input_scaled)
prediction


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 248ms/step


array([[0.21529667]], dtype=float32)

In [24]:
prediction_proba = prediction[0][0]
prediction_proba

np.float32(0.21529667)

In [25]:
if prediction_proba > 0.5:
    print("person is likely to churn")
else:
    print("person is not likely to churn")

person is not likely to churn
